# 기프트코 상품 추천 RAG v1
## row 단위 거래데이터 기반 추천 — 핵심 RAG 흐름 학습용

운영 품질을 높이는 개선 항목(수량별 단가 파싱, 추천 제외조건, 거래 가중치 고도화, Vector DB, Reranker 등)은
이번 버전 범위에서 제외했습니다. 핵심 흐름을 명확하게 보는 것이 우선입니다.

## 이 노트북에서 보는 RAG 4단계

| 단계 | RAG에서의 역할 | 이 노트북에서 하는 일 |
|---|---|---|
| **1. Indexing** | 검색 대상 문서를 임베딩으로 미리 변환 | 상품데이터/거래데이터를 텍스트로 정규화하고 임베딩 생성 |
| **2. Query 이해** | 사용자 질문에서 검색에 쓸 조건 추출 | LLM으로 질문을 구조화(JSON), 리드타임 반영한 참고월 계산 |
| **3. Retrieval** | 질문과 관련된 문서를 찾기 | 유사 거래 사례 검색 → 거래 신호 추출 → 상품 후보 랭킹 |
| **4. Generation** | 찾은 문서를 근거로 답변 생성 | 추천 후보 + 거래 신호를 컨텍스트로 LLM이 고객용 답변 작성 |


## v0와 달라진 점

| 구분 | v0 | v1 |
|---|---|---|
| 상품 검색 | TF-IDF 중심 | Ollama 임베딩 중심, 실패 시 TF-IDF fallback |
| 의도 확장 | 수동 사전 가능 | 수동 상품 사전 사용 안 함 |
| 거래데이터 | 약한 힌트 | 추천 방향의 핵심 신호 |
| 날짜 | 거의 미반영 | 행사/사용월 기준 1~2개월 앞선 주문월 반영 |
| 추천 점수 | 상품 텍스트 유사도 중심 | 거래 맥락 + 상품 임베딩 + 날짜 + 예산/수량 |
| AI 사용 | 선택적 답변 생성 | 질문 구조화 + 추천 설명 생성 가능 |

v1의 목표는 v0처럼 단순 TF-IDF나 수동 의도 사전에 의존하지 않고,  
**거래데이터의 구매 맥락과 날짜, 현재 상품데이터의 의미 유사도**를 함께 사용해 추천하는 것입니다.

## 핵심 전제

판촉물은 보통 행사 당월이 아니라 **사용/배포 예정일보다 1~2개월 앞서 주문**합니다.

```text
8월 행사/배포용
→ 거래데이터에서는 6~7월 주문 데이터가 더 중요
→ 8월 주문 데이터는 보조 참고
```

그래서 거래데이터의 `날짜`는 주문일자로, 사용자 질문의 월/계절은 사용·배포 예정 시점으로 해석합니다.

## 0. 설치

처음 한 번만 실행합니다.

```bash
python -m pip install pandas openpyxl scikit-learn numpy ollama
ollama pull bge-m3       # 임베딩 모델
ollama pull gemma3:4b    # 질문 구조화 / 답변 생성용 LLM
```

`ollama` 패키지가 없거나 `bge-m3` 임베딩이 실패하면 TF-IDF로 자동 대체(fallback)되도록 만들었습니다.

In [1]:
# 필요 시 한 번만 실행
# !pip install pandas openpyxl scikit-learn numpy ollama

## 1. 기본 설정 및 파일 경로

*RAG 단계: Indexing 준비*

In [2]:
from pathlib import Path
import re
import html
import json
import hashlib
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

# =========================
# 파일 경로
# =========================
PRODUCT_PATH = Path("../data/상품데이터_샘플100.xlsx")
TRADE_PATH = Path("../data/거래데이터_샘플100.xlsx")

OUTPUT_DIR = Path("../output")
CACHE_DIR = Path("../cache")
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# =========================
# 임베딩 / LLM 설정
# =========================
USE_OLLAMA_EMBEDDING = True
EMBED_MODEL = "bge-m3"       # 상품/거래 텍스트 임베딩 모델
LLM_MODEL = "gemma3:4b"      # 질문 구조화 및 답변 생성용 LLM

DEFAULT_TOP_K = 5            # 최종 추천 상품 개수

print("PRODUCT_PATH:", PRODUCT_PATH)
print("TRADE_PATH:", TRADE_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

PRODUCT_PATH: data\상품데이터_샘플100.xlsx
TRADE_PATH: data\거래데이터_샘플100.xlsx
OUTPUT_DIR: C:\Users\USER\Desktop\workspace\work-code-archive-organized\projects\rag-experiments\notebooks\product_recommendation\output


## 2. 파일 로딩 및 컬럼 확인

*RAG 단계: Indexing 준비 — 어떤 원본 데이터를 검색 대상으로 쓸지 확인합니다.*

In [3]:
product_raw = pd.read_excel(PRODUCT_PATH, dtype=str).fillna("")
trade_raw = pd.read_excel(TRADE_PATH, dtype=str).fillna("")

print("상품데이터 shape:", product_raw.shape)
print("거래데이터 shape:", trade_raw.shape)

print("\n[상품데이터 컬럼]")
for i, col in enumerate(product_raw.columns, start=1):
    print(f"{i:02d}. {col}")

print("\n[거래데이터 컬럼]")
for i, col in enumerate(trade_raw.columns, start=1):
    print(f"{i:02d}. {col}")

display(product_raw.head(3))
display(trade_raw.head(3))

상품데이터 shape: (100, 93)
거래데이터 shape: (100, 23)

[상품데이터 컬럼]
01. 일련번호
02. 상품번호
03. 입점업체ID
04. 브랜드
05. 상품명
06. 간략한설명
07. 모델명
08. 상품코드(바코드)
09. 상품판매가
10. 상품공급가
11. 수수료율
12. 시중가격
13. 네이버최저가설정
14. 적립금
15. 재고
16. 옵션명1
17. 옵션아이템1
18. 옵션1재고
19. 옵션1가격
20. 옵션명2
21. 옵션아이템2
22. 옵션명3
23. 옵션아이템3
24. 옵션명4
25. 옵션아이템4
26. 옵션명5
27. 옵션아이템5
28. 검색키워드
29. 대 카테고리
30. 중 카테고리
31. 소 카테고리
32. 세분류
33. 대 카테고리(중복)
34. 중 카테고리(중복)
35. 소 카테고리(중복)
36. 세분류(중복)
37. 큰이미지
38. 중간이미지
39. 작은이미지
40. 다른이미지
41. 상품내용
42. 과세/면세
43. 제조사
44. 원산지
45. 상품크기
46. 용량
47. 용량단위
48. 제목1
49. 내용1
50. 이벤트문구
51. 배송비구분
52. 배송비
53. 배송비무료금액
54. 배송정책
55. 최소구매수량설정
56. 최소구매수량
57. 구매단위
58. 복수구매
59. 복수구매수량계산방식
60. 첨부파일
61. 회원전용상품
62. 회원전용상품회원등급
63. 정보고시등록여부
64. 정보고시카테고리
65. 정보고시_1 내용
66. 정보고시_2 내용
67. 정보고시_3 내용
68. 정보고시_4 내용
69. 정보고시_5 내용
70. 정보고시_6 내용
71. 정보고시_7 내용
72. 정보고시_8 내용
73. 정보고시_9 내용
74. 정보고시_10 내용
75. 정보고시_11 내용
76. 정보고시_12 내용
77. 정보고시_13 내용
78. 정보고시_14 내용
79. 정보고시_15 내용
80. Unnamed: 79
81. Unnamed: 80
82. Unnamed: 81
83. 상품코드
84. 폐쇄몰공급가
85. 폐

,일련번호,상품번호,입점업체ID,브랜드,상품명,간략한설명,모델명,상품코드(바코드),상품판매가,상품공급가,수수료율,시중가격,네이버최저가설정,적립금,재고,옵션명1,옵션아이템1,옵션1재고,옵션1가격,옵션명2,옵션아이템2,옵션명3,옵션아이템3,옵션명4,옵션아이템4,옵션명5,옵션아이템5,검색키워드,대 카테고리,중 카테고리,소 카테고리,세분류,대 카테고리(중복),중 카테고리(중복),소 카테고리(중복),세분류(중복),큰이미지,중간이미지,작은이미지,다른이미지,상품내용,과세/면세,제조사,원산지,상품크기,용량,용량단위,제목1,내용1,이벤트문구,배송비구분,배송비,배송비무료금액,배송정책,최소구매수량설정,최소구매수량,구매단위,복수구매,복수구매수량계산방식,첨부파일,회원전용상품,회원전용상품회원등급,정보고시등록여부,정보고시카테고리,정보고시_1 내용,정보고시_2 내용,정보고시_3 내용,정보고시_4 내용,정보고시_5 내용,정보고시_6 내용,정보고시_7 내용,정보고시_8 내용,정보고시_9 내용,정보고시_10 내용,정보고시_11 내용,정보고시_12 내용,정보고시_13 내용,정보고시_14 내용,정보고시_15 내용,Unnamed: 79,Unnamed: 80,Unnamed: 81,상품코드,폐쇄몰공급가,폐쇄몰판매가,해외판매공급가,리워드판매수익퍼센트,리워드소개수익퍼센트,Unnamed: 88,Unnamed: 89,Unnamed: 90,Unnamed: 91,Unnamed: 92
0,23271,36601,osiris0000,,[PQI] USB 메모리 U273V USB3.0 8GB~,,,,10925,9500,60,0,,0,,,,,,,,,,,,,,"pqi,usb,메모리,usb메모리,외장하드,외장디스크,pqiusb",가전/디지털/USB,USB메모리,캡방식USB,,,,,,08/14750008.jpg,08/14750008.jpg,08/14750008.jpg,,"<img src=""../../data/prmongddang/goods/productimg/detail/img/65/P78128765/P78128765_1.jpg"" border=""0"" />",과세,상담 문의,중국,,53.8*17.2*8.6mm,,,,,,0,0,,설정함,1000,1,"10:13300,30:12920,50:12540,100:11780,300:11495,500:11210,1000:10925",개별,,,,무,,,,,,,,,,,,,,,,,,,,14750008,0,0,0,0,0,,,,,
1,11571,79362,thegreen,,오버핏 라운드 티셔츠,,,,6435,5500,0,0,,0,,,,,,,,,,,,,,#티셔츠#라운드#레이어드,단체복,티셔츠,가방/의류,,,,,,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,오버핏-라운드-티셔츠(17수)-썸네일(500)-수정.jpg,,"<div style=""text-align: left;"">&nbsp;</div> <p><a data-goodscontent=""contentimg""><img src=""../../data/prmongddang/editor/202309/오버핏-라운드-티셔츠(17수)-상세페이지.jpg"" /></a></p>",과세,상담 문의,상담 문의,,,,,,,,0,0,,설정함,5000,1,"50:7205,100:6985,300:6875,500:6765,1000:6655,3000:6545,5000:6435",전체,,,,무,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2,22636,39442,hyuga11,,오로라노크 볼펜,,,,175,155,60,0,,0,,,,,,,,,,,,,,,볼펜/사무/문구,볼펜/필기류,100원~500원,,,,,,10/15440210.jpg,10/15440210.jpg,10/15440210.jpg,,"<img src=""http://prmongddang.com/data/prmongddang/goods/productimg/detail/img/10/15440210/15440210_1.jpg"" border=""0"" />",과세,상담 문의,중국,,152mm*9mm,,,,,,0,0,,설정함,30000,1,"500:214,1000:205,3000:195,5000:189,10000:181,20000:177,30000:175",개별,,,,무,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,


,번호,구매처 분류(대),구매처 분류(중),구매처 분류(소),구매처 분류(세),구매처 명,날짜,상품,상품분류(대),상품분류(중),상품분류(소),대량가격(원),중간가격(원),소량가격(원),최소구매수량,행사별(필터),대상별(필터),시즌별(필터),인쇄 방법,비 고,_출처폴더,_출처파일,구매처 명
0,702,기업/회사,제약/의료기기/바이오,의료기기/의료용품,의료기기/의료용품,파라곤CRT,2023/09/25,쏘쏘 원터치 스텐텀블러 500ml 1P,일상용품,주방용품,저장용기,4500,4578,5374,20,,,,칼라전사필름인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_제약의료기기바이오_完(973).xlsx,
1,,기업/회사,일반기업/회사,화장품/패션/주얼리,화장품,Changefit,2021/12/14,귀요미 캐릭터 스마트링 1P,디지털/가전,휴대폰액세서리,기타휴대폰액세서리,850,873,989,50,,,,칼라인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_일반기업회사_完(4909).xlsx,
2,,기업/회사,일반기업/회사,외국계회사/글로벌기업,외국계회사/글로벌기업,CONMAD,2022/06/27,에스모도 미니 핸디형 선풍기 159,디지털/가전,건강/냉난방가전,선풍기,7240,7556,8458,10,,,,칼라인쇄,,기업회사(16692),판촉물_거래_데이터_조사_자료_기업회사_일반기업회사_完(4909).xlsx,


## 3. 공통 전처리 함수

*RAG 단계: Indexing 준비 — 텍스트 정제, 벡터 정규화/유사도 계산에 쓰는 유틸리티입니다.*

In [4]:
def clean_text(x):
    """HTML 태그/엔티티를 제거하고 공백을 정리합니다."""
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = html.unescape(x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def to_number(x):
    """'3,000원' 같은 문자열에서 숫자만 뽑아 float로 변환합니다."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "nan", "None", "null"]:
        return np.nan
    s = re.sub(r"[^0-9.]", "", s)
    if s == "":
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan


def safe_col(df, col):
    """컬럼이 없으면 빈 문자열 시리즈를 돌려줘서 KeyError를 막습니다."""
    if col in df.columns:
        return df[col].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def combine_text(*parts):
    return clean_text(" ".join([str(p) for p in parts if str(p).strip()]))


def normalize_vector_matrix(mat: np.ndarray) -> np.ndarray:
    """임베딩 행렬을 행 단위 L2 정규화합니다 (코사인 유사도를 내적으로 계산하기 위함)."""
    mat = np.array(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return mat / norms


def cosine_scores(query_vec: np.ndarray, matrix: np.ndarray) -> np.ndarray:
    """정규화된 query 벡터와 문서 행렬 간 코사인 유사도(=내적)를 계산합니다."""
    query_vec = np.array(query_vec, dtype=np.float32)
    if query_vec.ndim == 1:
        query_vec = query_vec.reshape(1, -1)
    query_norm = np.linalg.norm(query_vec, axis=1, keepdims=True)
    query_norm[query_norm == 0] = 1
    query_vec = query_vec / query_norm
    return (matrix @ query_vec.T).ravel()


def text_hash(texts):
    """임베딩 캐시 파일명을 만들기 위한 내용 해시입니다."""
    joined = "\n".join([str(x) for x in texts])
    return hashlib.md5(joined.encode("utf-8")).hexdigest()[:10]

## 4. 상품데이터 정규화

*RAG 단계: Indexing — 나중에 임베딩할 "문서"를 상품 1건당 1개씩 만듭니다.*

상품 검색용 텍스트(`product_search_text`)에는 아래를 포함합니다.

```text
상품명 / 브랜드 / 모델명 / 대·중·소 카테고리 / 검색키워드 / 설명 / 제조사·원산지
```

In [5]:
product_df = product_raw.copy()

product_df["product_id"] = safe_col(product_df, "상품번호")
product_df.loc[product_df["product_id"].str.strip() == "", "product_id"] = safe_col(product_df, "상품코드")

product_df["brand"] = safe_col(product_df, "브랜드").map(clean_text)
product_df["product_name"] = safe_col(product_df, "상품명").map(clean_text)
product_df["model_name"] = safe_col(product_df, "모델명").map(clean_text)

product_df["price"] = safe_col(product_df, "상품판매가").map(to_number)
product_df["moq"] = safe_col(product_df, "최소구매수량").map(to_number)

product_df["category_large"] = safe_col(product_df, "대 카테고리").map(clean_text)
product_df["category_middle"] = safe_col(product_df, "중 카테고리").map(clean_text)
product_df["category_small"] = safe_col(product_df, "소 카테고리").map(clean_text)
product_df["category_detail"] = safe_col(product_df, "세분류").map(clean_text)

product_df["category_path"] = (
    product_df["category_large"] + " > " +
    product_df["category_middle"] + " > " +
    product_df["category_small"] + " > " +
    product_df["category_detail"]
).map(clean_text)

product_df["keywords"] = safe_col(product_df, "검색키워드").map(clean_text)
product_df["summary"] = safe_col(product_df, "간략한설명").map(clean_text)
product_df["detail_text"] = safe_col(product_df, "상품내용").map(clean_text)
product_df["manufacturer"] = safe_col(product_df, "제조사").map(clean_text)
product_df["origin"] = safe_col(product_df, "원산지").map(clean_text)
product_df["image"] = safe_col(product_df, "큰이미지").map(clean_text)

# 임베딩에 넣을 "문서" 텍스트 — 이 텍스트가 곧 검색 대상입니다.
product_df["product_search_text"] = (
    "[상품명] " + product_df["product_name"] + "\n" +
    "[브랜드] " + product_df["brand"] + "\n" +
    "[모델명] " + product_df["model_name"] + "\n" +
    "[카테고리] " + product_df["category_path"] + "\n" +
    "[검색키워드] " + product_df["keywords"] + "\n" +
    "[설명] " + product_df["summary"] + " " + product_df["detail_text"] + "\n" +
    "[제조/원산지] " + product_df["manufacturer"] + " " + product_df["origin"]
).map(clean_text)

product_df = product_df[product_df["product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 상품 수:", len(product_df))
display(product_df[["product_id", "product_name", "price", "moq", "category_path", "keywords"]].head(10))

정규화 상품 수: 100


,product_id,product_name,price,moq,category_path,keywords
0,36601,[PQI] USB 메모리 U273V USB3.0 8GB~,10925.0,1000.0,가전/디지털/USB > USB메모리 > 캡방식USB >,"pqi,usb,메모리,usb메모리,외장하드,외장디스크,pqiusb"
1,79362,오버핏 라운드 티셔츠,6435.0,5000.0,단체복 > 티셔츠 > 가방/의류 >,#티셔츠#라운드#레이어드
2,39442,오로라노크 볼펜,175.0,30000.0,볼펜/사무/문구 > 볼펜/필기류 > 100원~500원 >,
3,87400,[ALIO] 모던클락 15W 우드형 LED시계&고속무선충전기,12430.0,3000.0,스마트폰용품 > 휴대폰 무선충전패드 > 가전/디지털/USB >,#led시계 #시계 #우드시계 #나무시계 #디지털시계 #고속무선충전패드 #충전패드 #무선충전 #무선충전패드 #무선충전시계 #시계패드추천 #무선패드 #고속무선시계
4,101669,[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈),4163.0,5000.0,가정/생활용품 > 치약칫솔세트외 > 치약/칫솔세트 >,#선물세트 #유시몰 #크리넥스 #칫솔치약세트 #물티슈
5,31300,패밀리여행용세트 대1,23595.0,3000.0,레저/건강/차량 > 여행용품 > 여행용세면도구 >,#여행세트#여행용세트#세면세트
6,83327,국산 스페이스 포켓 시장가방,2825.0,10000.0,에코백/가방 > 장바구니 > 가방/의류 >,"#국산가방, #시장가방, #장바구니"
7,85324,[크로커다일] 3단 65-10K 우드 곡자(자동) 우산,16215.0,1000.0,가정/생활용품 > 우산/우의 > 3단우산 >,#우산 #3단우산#자동우산#크로커다일
8,3814,메탈레드 30매 물티슈,263.0,100000.0,가정/생활용품 > 물티슈/휴지 > 30매 이상 물티슈 >,"물휴지,물티슈,티슈,전도용물티슈,홍보용물티슈,휴대용물티슈,광고,판촉,전도티슈,물티슈,홍보용티슈,광고용물티슈,광고물티슈,홍보용,판촉물,수미향"
9,92474,[참그린] 매실주방세제펌프 피죤 베이킹소다300리필 3종세트,4344.0,3000.0,주방/식품/텀블러 > 주방/세탁 세제 > 주방/세탁 혼합세트 >,"주방세제, 1종 주방세제, 친환경 주방세제, 무료배송,세제선물세트,펌프주방세제,설거지, 세제,매실,퐁퐁, 식기세척, 트리오"


## 5. 거래데이터 정규화 (row 단위)

*RAG 단계: Indexing — 거래데이터는 추천 대상 DB가 아니라, "과거에 비슷한 상황에서 무엇을 샀는지" 찾기 위한
보조 검색 대상입니다. 거래 1건(row)을 문서 1개로 취급합니다 (→ profile 단위 집계는 이번 버전 범위 밖).*

In [6]:
trade_df = trade_raw.copy()

buyer_name_col = "구매처 명 "
if buyer_name_col not in trade_df.columns and "구매처 명" in trade_df.columns:
    buyer_name_col = "구매처 명"

trade_df["buyer_type_large"] = safe_col(trade_df, "구매처 분류(대)").map(clean_text)
trade_df["buyer_type_middle"] = safe_col(trade_df, "구매처 분류(중)").map(clean_text)
trade_df["buyer_type_small"] = safe_col(trade_df, "구매처 분류(소)").map(clean_text)
trade_df["buyer_type_detail"] = safe_col(trade_df, "구매처 분류(세)").map(clean_text)
trade_df["buyer_name"] = safe_col(trade_df, buyer_name_col).map(clean_text)

trade_df["buyer_type_path"] = (
    trade_df["buyer_type_large"] + " > " +
    trade_df["buyer_type_middle"] + " > " +
    trade_df["buyer_type_small"] + " > " +
    trade_df["buyer_type_detail"]
).map(clean_text)

# 거래데이터의 '날짜' = 주문일자로 해석합니다.
trade_df["date_raw"] = safe_col(trade_df, "날짜").map(clean_text)
trade_df["order_date"] = pd.to_datetime(trade_df["date_raw"], errors="coerce")
trade_df["order_month"] = trade_df["order_date"].dt.month

trade_df["trade_product_name"] = safe_col(trade_df, "상품").map(clean_text)

trade_df["trade_category_large"] = safe_col(trade_df, "상품분류(대)").map(clean_text)
trade_df["trade_category_middle"] = safe_col(trade_df, "상품분류(중)").map(clean_text)
trade_df["trade_category_small"] = safe_col(trade_df, "상품분류(소)").map(clean_text)

trade_df["trade_category_path"] = (
    trade_df["trade_category_large"] + " > " +
    trade_df["trade_category_middle"] + " > " +
    trade_df["trade_category_small"]
).map(clean_text)

trade_df["bulk_price"] = safe_col(trade_df, "대량가격(원)").map(to_number)
trade_df["middle_price"] = safe_col(trade_df, "중간가격(원)").map(to_number)
trade_df["small_price"] = safe_col(trade_df, "소량가격(원)").map(to_number)
trade_df["trade_moq"] = safe_col(trade_df, "최소구매수량").map(to_number)

trade_df["event_filter"] = safe_col(trade_df, "행사별(필터)").map(clean_text)
trade_df["target_filter"] = safe_col(trade_df, "대상별(필터)").map(clean_text)
trade_df["season_filter"] = safe_col(trade_df, "시즌별(필터)").map(clean_text)
trade_df["print_method"] = safe_col(trade_df, "인쇄 방법").map(clean_text)
trade_df["memo"] = safe_col(trade_df, "비  고").map(clean_text)

# 거래 1건(row)을 하나의 검색 문서로 만듭니다.
trade_df["trade_search_text"] = (
    "[구매처] " + trade_df["buyer_type_path"] + " " + trade_df["buyer_name"] + "\n" +
    "[주문월] " + trade_df["order_month"].fillna("").astype(str) + "월\n" +
    "[상품] " + trade_df["trade_product_name"] + "\n" +
    "[상품분류] " + trade_df["trade_category_path"] + "\n" +
    "[행사/대상/시즌] " + trade_df["event_filter"] + " " + trade_df["target_filter"] + " " + trade_df["season_filter"] + "\n" +
    "[인쇄] " + trade_df["print_method"] + "\n" +
    "[비고] " + trade_df["memo"]
).map(clean_text)

trade_df = trade_df[trade_df["trade_product_name"].str.len() > 0].reset_index(drop=True)

print("정규화 거래 수 (row 단위):", len(trade_df))
display(trade_df[[
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "bulk_price", "middle_price", "small_price", "trade_moq",
    "event_filter", "target_filter", "season_filter",
]].head(10))

정규화 거래 수 (row 단위): 100


,date_raw,order_month,buyer_type_path,buyer_name,trade_product_name,trade_category_path,bulk_price,middle_price,small_price,trade_moq,event_filter,target_filter,season_filter
0,2023/09/25,9.0,기업/회사 > 제약/의료기기/바이오 > 의료기기/의료용품 > 의료기기/의료용품,파라곤CRT,쏘쏘 원터치 스텐텀블러 500ml 1P,일상용품 > 주방용품 > 저장용기,4500.0,4578.0,5374.0,20.0,,,
1,2021/12/14,12.0,기업/회사 > 일반기업/회사 > 화장품/패션/주얼리 > 화장품,Changefit,귀요미 캐릭터 스마트링 1P,디지털/가전 > 휴대폰액세서리 > 기타휴대폰액세서리,850.0,873.0,989.0,50.0,,,
2,2022/06/27,6.0,기업/회사 > 일반기업/회사 > 외국계회사/글로벌기업 > 외국계회사/글로벌기업,CONMAD,에스모도 미니 핸디형 선풍기 159,디지털/가전 > 건강/냉난방가전 > 선풍기,7240.0,7556.0,8458.0,10.0,,,
3,44348,NaN,학교/교육기관 > 대학교/대학원 > 취업/창업지원/일자리센터 > 취업/창업지원/일자리센터,성결대학교 대학일자리센터,(완판) [카카오프렌즈] 코마사 세면타월 160g 1P,일상용품 > 욕실/청소용품 > 타월,NaN,NaN,NaN,NaN,,,
4,2025/01/08,1.0,기업/회사 > 전문직 > 세무사/회계법인 > 세무사/회계법인,신원세무회계,2025년 포인트세법 탁상달력 캘린더 카렌다 250*190mm 1P,교육/문화용품 > 문구/사무용품 > 종이류,990.0,1135.0,1523.0,100.0,,,
5,2023/01/18,1.0,기업/회사 > 일반기업/회사 > 외국계회사/글로벌기업 > 외국계회사/글로벌기업,MYNAVI KOREA CORPORATION,(인쇄 무료 이벤트) [마이브렐라] C형 수동 거꾸로 장우산 1P,패션잡화 > 패션소품 > 우산,8440.0,8730.0,9894.0,10.0,,,
6,2024/02/22,2.0,문화시설/스포츠관련 > 골프관련 > 국립공원/수목원 > 국립공원/수목원,JIRISAN DULLEBOGO,[뮤스트] 팝스 USB 2.0 메모리 (4GB~128GB) 1P,디지털/가전 > 저장장치 > 스토리지,3800.0,3948.0,4278.0,20.0,,,
7,2021/03/22,3.0,기업/회사 > 일반기업/회사 > 식품회사/건강식품 > 식품회사/건강식품,JAWS,국내산쿨마스크 일반형 패션마스크 1P,일상용품 > 생활용품 > 생활잡화류,830.0,NaN,NaN,NaN,,,
8,2023/10/05,10.0,관공서/공공기관 > 보건소/정신건강/보건복지관련 > 보건소 > 보건소,부안군보건소,[울트라웨이브] 휴대용 마스크 살균건조기 MS-01 1P,디지털/가전 > 생활가전 > 건조기/탈수기,16000.0,16296.0,18139.0,5.0,,,
9,2021/03/17,3.0,학교/교육기관 > 중고등학교 > 고등학교 > 고등학교,계명고등학교,전도 종이투명칼라 지갑티슈 (10조 20매) 1P,일상용품 > 위생용품 > 화장지,144.0,147.0,196.0,2000.0,,,


## 6. 질문 조건 추출

*RAG 단계: Query 이해 — 검색에 쓸 조건(예산/수량/시즌/월)을 질문에서 뽑아냅니다.*

두 가지 방식을 씁니다.

1. **Ollama LLM 조건 추출** — 구매처/행사/목적/계절/월/예산/수량을 JSON으로 추출
2. **정규식 fallback** — Ollama가 없거나 실패할 때 예산/수량/월/계절만 간단히 추출

수동 상품 사전은 쓰지 않습니다 (예: "여름이면 선풍기" 같은 임의 확장 금지). 계절→월 범위만 달력 개념으로 사용합니다.

In [7]:
SEASON_MONTHS = {
    "봄": [3, 4, 5],
    "여름": [6, 7, 8],
    "가을": [9, 10, 11],
    "겨울": [12, 1, 2],
}


def extract_basic_conditions(query: str) -> dict:
    """정규식만으로 예산/수량/월/계절을 추출하는 fallback입니다."""
    q = str(query)

    budget = None
    m = re.search(r"(\d+(?:,\d+)*)\s*원", q)
    if m:
        budget = int(m.group(1).replace(",", ""))
    if budget is None:
        m = re.search(r"(\d+)\s*천\s*원", q)
        if m:
            budget = int(m.group(1)) * 1000
    if budget is None:
        m = re.search(r"(\d+)\s*만\s*원", q)
        if m:
            budget = int(m.group(1)) * 10000

    quantity = None
    m = re.search(r"(\d+(?:,\d+)*)\s*(?:개|ea|EA|pcs|PCS)", q)
    if m:
        quantity = int(m.group(1).replace(",", ""))

    event_month = None
    m = re.search(r"(1[0-2]|[1-9])\s*월", q)
    if m:
        event_month = int(m.group(1))

    season = next((s for s in SEASON_MONTHS if s in q), None)

    return {
        "raw_query": q,
        "buyer_context": "",
        "event_context": "",
        "purpose": "",
        "season": season,
        "event_month": event_month,
        "budget_max": budget,
        "quantity": quantity,
        "style_keywords": [],
        "must_have": [],
        "extraction_method": "regex_fallback",
    }


def safe_json_loads(text: str) -> dict:
    """LLM 응답에서 코드블록/잡음을 제거하고 JSON 객체만 파싱합니다."""
    text = str(text).strip()
    text = re.sub(r"^```json", "", text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$", "", text).strip()
    start, end = text.find("{"), text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def extract_query_condition(query: str, model: str = LLM_MODEL) -> dict:
    """
    Ollama LLM으로 질문을 구조화합니다. 실패하면 정규식 fallback을 씁니다.
    이 함수의 반환값(query_condition)이 이후 모든 검색/랭킹 단계의 입력이 됩니다.
    """
    fallback = extract_basic_conditions(query)

    try:
        import ollama
    except ModuleNotFoundError:
        return fallback

    system_prompt = """
너는 판촉물 상품 추천을 위한 조건 추출기다.
사용자 질문에서 추천에 필요한 조건만 JSON으로 추출한다.

주의:
- 상품 추천 사전을 만들지 않는다.
- "여름이면 선풍기"처럼 상품군을 임의 확장하지 않는다.
- 사용자가 말한 구매처, 행사, 목적, 계절, 월, 예산, 수량, 스타일만 구조화한다.
- 모르는 값은 null 또는 빈 문자열/빈 리스트로 둔다.
- 반드시 JSON만 출력한다.
"""

    user_prompt = f"""
아래 사용자 요청을 JSON으로 구조화해줘.

[사용자 요청]
{query}

[출력 JSON 스키마]
{{
  "buyer_context": "구매처/업종/기관/대상 예: 병원, 대학교, 기업, 어린이집",
  "event_context": "행사/상황 예: 개원, 창립기념, 박람회, 여름행사",
  "purpose": "용도 예: 답례품, 사은품, 홍보물, 기념품",
  "season": "봄/여름/가을/겨울 중 하나 또는 null",
  "event_month": 1~12 숫자 또는 null,
  "budget_max": 숫자 또는 null,
  "quantity": 숫자 또는 null,
  "style_keywords": ["실용적", "고급", "저렴한"] 형태,
  "must_have": ["로고인쇄", "휴대성"] 형태
}}
"""

    try:
        response = ollama.chat(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt.strip()},
                {"role": "user", "content": user_prompt.strip()},
            ],
            options={"temperature": 0.0, "num_ctx": 2048},
            stream=False,
        )
        parsed = safe_json_loads(response["message"]["content"])

        result = fallback.copy()
        for k, v in parsed.items():
            if k in result:
                result[k] = v

        # 정규식이 잡은 예산/수량/월/계절을 LLM이 놓쳤으면 보존
        for k in ["budget_max", "quantity", "event_month", "season"]:
            if result.get(k) in [None, "", []] and fallback.get(k) not in [None, "", []]:
                result[k] = fallback[k]

        result["raw_query"] = query
        result["extraction_method"] = "ollama_llm"
        return result

    except Exception as e:
        fallback["extraction_error"] = str(e)
        return fallback


# 테스트
for q in [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
]:
    print("\nQUERY:", q)
    print(extract_query_condition(q))


QUERY: 8월 행사에서 나눠줄 여름 판촉물 추천해줘
{'raw_query': '8월 행사에서 나눠줄 여름 판촉물 추천해줘', 'buyer_context': None, 'event_context': '여름 행사', 'purpose': '판촉물', 'season': '여름', 'event_month': 8, 'budget_max': None, 'quantity': None, 'style_keywords': [], 'must_have': [], 'extraction_method': 'ollama_llm'}

QUERY: 병원 개원 답례품으로 3천원 이하 500개 추천해줘
{'raw_query': '병원 개원 답례품으로 3천원 이하 500개 추천해줘', 'buyer_context': '병원', 'event_context': '개원', 'purpose': '답례품', 'season': None, 'event_month': None, 'budget_max': 3000, 'quantity': 500, 'style_keywords': [], 'must_have': [], 'extraction_method': 'ollama_llm'}

QUERY: 대학교 OT에서 나눠줄 저렴한 사은품 추천해줘
{'raw_query': '대학교 OT에서 나눠줄 저렴한 사은품 추천해줘', 'buyer_context': '대학교', 'event_context': 'OT', 'purpose': '사은품', 'season': None, 'event_month': None, 'budget_max': None, 'quantity': None, 'style_keywords': ['저렴한'], 'must_have': [], 'extraction_method': 'ollama_llm'}


## 7. 행사/사용월 기준 참고 주문월 계산

*RAG 단계: Query 이해 — 도메인 지식(판촉물은 1~2개월 앞서 주문)을 검색 가중치로 바꿉니다.*

예: 8월 행사 → 7월 주문(가중치 1.0) · 6월 주문(0.85) · 8월 주문(0.45, 보조) 순으로 참고합니다.

In [8]:
def prev_month(month: int, n: int = 1) -> int:
    m = month - n
    while m <= 0:
        m += 12
    return m


def get_month_weights(event_month: int) -> dict:
    """사용/행사월 기준으로 '주문월이 몇 월이면 얼마나 중요한지' 가중치를 만듭니다."""
    if event_month is None or pd.isna(event_month):
        return {}
    event_month = int(event_month)
    return {
        prev_month(event_month, 1): 1.00,
        prev_month(event_month, 2): 0.85,
        event_month: 0.45,
        prev_month(event_month, 3): 0.25,
    }


def get_event_months(query_condition: dict) -> list:
    """query_condition에서 '사용/행사 시점 후보 월' 목록을 뽑습니다 (event_month 단일값 + season 범위)."""
    months = []
    event_month = query_condition.get("event_month")
    if event_month not in [None, "", np.nan]:
        try:
            months.append(int(event_month))
        except Exception:
            pass

    season = query_condition.get("season")
    if season in SEASON_MONTHS:
        months.extend(SEASON_MONTHS[season])

    return sorted(set(months))


def get_reference_month_weights(query_condition: dict) -> dict:
    """여러 후보 월의 가중치를 월별로 합쳐서(최댓값) 하나의 참고월 가중치 테이블로 만듭니다."""
    combined = {}
    for m in get_event_months(query_condition):
        for month, score in get_month_weights(m).items():
            combined[month] = max(combined.get(month, 0), score)
    return combined


def calc_date_lead_score(order_month, query_condition: dict) -> float:
    """거래 1건의 주문월이 참고월 가중치 테이블에서 몇 점인지 돌려줍니다."""
    if pd.isna(order_month):
        return 0.0
    weights = get_reference_month_weights(query_condition)
    if not weights:
        return 0.0
    try:
        return float(weights.get(int(order_month), 0.0))
    except Exception:
        return 0.0


# 테스트
for q in ["8월 행사 판촉물", "여름 행사 판촉물", "12월 연말 선물"]:
    query_condition = extract_basic_conditions(q)
    print(q, "->", get_reference_month_weights(query_condition))

8월 행사 판촉물 -> {7: 1.0, 6: 0.85, 8: 0.45, 5: 0.25}
여름 행사 판촉물 -> {5: 1.0, 4: 0.85, 6: 1.0, 3: 0.25, 7: 1.0, 8: 0.45}
12월 연말 선물 -> {11: 1.0, 10: 0.85, 12: 0.45, 9: 0.25}


## 8. 임베딩 생성

*RAG 단계: Retrieval (인덱스 구축) — 상품데이터/거래데이터 문서를 벡터로 변환합니다.*

기본은 Ollama `bge-m3` 임베딩이고, 실패하면 자동으로 TF-IDF fallback을 씁니다.

In [9]:
def try_ollama_embed_texts(texts, model=EMBED_MODEL, batch_size=32):
    """Ollama Python 패키지의 embed()/embeddings() API 차이를 모두 지원합니다."""
    import ollama

    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch = [str(x) for x in texts[start:start + batch_size]]
        try:
            resp = ollama.embed(model=model, input=batch)
            if "embeddings" in resp:
                all_embeddings.extend(resp["embeddings"])
                continue
        except Exception:
            pass

        for text in batch:
            try:
                resp = ollama.embeddings(model=model, prompt=text)
                all_embeddings.append(resp["embedding"])
            except Exception as e:
                raise RuntimeError(f"Ollama embedding 실패: {e}")

    return np.array(all_embeddings, dtype=np.float32)


def build_search_index(name, texts, use_ollama=True, model=EMBED_MODEL):
    """
    임베딩 인덱스를 만듭니다. 캐시가 있으면 재사용합니다.
    반환: {"backend": "ollama"|"tfidf", "matrix": ..., "vectorizer": ...}
    """
    texts = [str(x) for x in texts]
    cache_path = CACHE_DIR / f"{name}_{model}_{text_hash(texts)}.npy"

    if use_ollama:
        try:
            if cache_path.exists():
                matrix = normalize_vector_matrix(np.load(cache_path))
                print(f"[{name}] 임베딩 캐시 로딩: {cache_path.name}")
                return {"backend": "ollama", "model": model, "matrix": matrix, "vectorizer": None}

            print(f"[{name}] Ollama 임베딩 생성 중... rows={len(texts)}")
            matrix = try_ollama_embed_texts(texts, model=model)
            np.save(cache_path, matrix)
            return {"backend": "ollama", "model": model, "matrix": normalize_vector_matrix(matrix), "vectorizer": None}

        except Exception as e:
            warnings.warn(f"[{name}] Ollama 임베딩 실패, TF-IDF로 대체합니다: {e}")

    print(f"[{name}] TF-IDF fallback 생성")
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=1)
    matrix = vectorizer.fit_transform(texts)
    return {"backend": "tfidf", "model": "tfidf_char_ngram", "matrix": matrix, "vectorizer": vectorizer}


def search_index(query_text, index) -> np.ndarray:
    """인덱스 backend에 맞춰 query와 문서들 간 유사도 점수를 계산합니다."""
    if index["backend"] == "ollama":
        q_vec = normalize_vector_matrix(try_ollama_embed_texts([query_text], model=index["model"]))
        return cosine_scores(q_vec[0], index["matrix"])
    if index["backend"] == "tfidf":
        q_vec = index["vectorizer"].transform([query_text])
        return cosine_similarity(q_vec, index["matrix"]).flatten()
    raise ValueError(f"지원하지 않는 backend: {index['backend']}")


product_index = build_search_index("product", product_df["product_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)
trade_index = build_search_index("trade", trade_df["trade_search_text"], use_ollama=USE_OLLAMA_EMBEDDING)

print("product backend:", product_index["backend"])
print("trade backend:", trade_index["backend"])

[product] Ollama 임베딩 생성 중... rows=100
[trade] Ollama 임베딩 생성 중... rows=100
product backend: ollama
trade backend: ollama


## 9. 유사 거래 사례 검색 (row 단위)

*RAG 단계: Retrieval — 질문과 비슷한 거래 사례를 찾습니다. 의미 유사도만 보지 않고
날짜 리드타임 점수, 예산/수량/시즌 조건을 함께 반영합니다.*

```text
trade_total_score = 의미유사도(55%) + 날짜리드타임(25%) + 예산/수량/시즌 조건(20%)
```

In [10]:
def build_query_text(query_condition: dict) -> str:
    """query_condition(구조화된 질문)을 검색 쿼리 문자열로 합칩니다."""
    parts = [
        query_condition.get("raw_query", ""),
        query_condition.get("buyer_context", ""),
        query_condition.get("event_context", ""),
        query_condition.get("purpose", ""),
        query_condition.get("season", ""),
        " ".join(query_condition.get("style_keywords") or []),
        " ".join(query_condition.get("must_have") or []),
    ]
    return combine_text(*parts)


def calc_trade_condition_score(row, query_condition: dict) -> float:
    """예산/수량/시즌이 이 거래 건과 얼마나 맞는지 0~1 점수로 계산합니다."""
    score = 0.0

    budget = query_condition.get("budget_max")
    if budget not in [None, ""] and not pd.isna(budget):
        prices = [p for p in [row.get("bulk_price"), row.get("middle_price"), row.get("small_price")] if not pd.isna(p)]
        if prices:
            min_price = min(prices)
            if min_price <= float(budget):
                score += 0.35
            elif min_price <= float(budget) * 1.2:
                score += 0.15

    quantity = query_condition.get("quantity")
    if quantity not in [None, ""] and not pd.isna(quantity):
        moq = row.get("trade_moq", np.nan)
        if pd.isna(moq) or moq <= float(quantity):
            score += 0.25

    season = query_condition.get("season")
    if season and season in str(row.get("season_filter", "")):
        score += 0.20

    return min(score, 1.0)


def retrieve_similar_trades(query: str, query_condition: dict, top_k: int = 20) -> pd.DataFrame:
    """질문과 가장 관련 있는 거래 사례 top_k건을 점수와 함께 돌려줍니다."""
    query_text = build_query_text(query_condition)
    semantic_scores = search_index(query_text, trade_index)

    result = trade_df.copy()
    result["trade_semantic_score"] = semantic_scores
    result["date_lead_score"] = result["order_month"].apply(lambda m: calc_date_lead_score(m, query_condition))
    result["trade_condition_score"] = result.apply(lambda row: calc_trade_condition_score(row, query_condition), axis=1)

    result["trade_total_score"] = (
        result["trade_semantic_score"] * 0.55 +
        result["date_lead_score"] * 0.25 +
        result["trade_condition_score"] * 0.20
    )

    return result.sort_values("trade_total_score", ascending=False).head(top_k).reset_index(drop=True)


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
test_condition = extract_query_condition(test_query)
similar_trades = retrieve_similar_trades(test_query, test_condition, top_k=10)

print("[추출된 조건]")
print(json.dumps(test_condition, ensure_ascii=False, indent=2))

display(similar_trades[[
    "trade_total_score", "trade_semantic_score", "date_lead_score", "trade_condition_score",
    "date_raw", "order_month", "buyer_type_path", "buyer_name",
    "trade_product_name", "trade_category_path",
    "event_filter", "target_filter", "season_filter",
]])

[추출된 조건]
{
  "raw_query": "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
  "buyer_context": null,
  "event_context": "여름 행사",
  "purpose": "판촉물",
  "season": "여름",
  "event_month": 8,
  "budget_max": null,
  "quantity": null,
  "style_keywords": [],
  "must_have": [],
  "extraction_method": "ollama_llm"
}


,trade_total_score,trade_semantic_score,date_lead_score,trade_condition_score,date_raw,order_month,buyer_type_path,buyer_name,trade_product_name,trade_category_path,event_filter,target_filter,season_filter
0,0.513035,0.478246,1.0,0.0,2021/06/10,6.0,센터복지관련기관 > 청년관련센터/기관 > 청년관련기관센터 기타 > 청년관련기관센터 기타,NARAE,[캐나믹스] CANAMIX M6 텀블러 600ml 1P,일상용품 > 주방용품 > 저장용기,,,
1,0.507466,0.468120,1.0,0.0,2023/05/19,5.0,기업/회사 > 제조/생산기업 > 금속/철강/부품 > 금속/철강/부품,ALLIED MINERAL PROCUCTS,[파카] 조터 오리지널 볼펜 (스페셜),교육/문화용품 > 문구/사무용품 > 필기용품,,,
2,0.504066,0.461937,1.0,0.0,2022/07/27,7.0,기업/회사 > 부동산/분양/모델하우스 > 분양/모델하우스 > 분양/모델하우스,롯데캐슬 드메르,미니손톱깎이 (부엉이),일상용품 > 생활용품 > 이미용소품,,,
3,0.497600,0.450182,1.0,0.0,2021/05/07,5.0,학교/교육기관 > 중고등학교 > 고등학교 > 고등학교,충북반도체고등학교,노트북 파우치 가방 14.1(39*28*4.0cm),디지털/가전 > 노트북액세서리 > 노트북가방/케이스,,,
4,0.496401,0.448001,1.0,0.0,2024/06/28,6.0,관공서/공공기관 > 경찰청/경찰서/소방서 > 소방서 > 소방서,119청소년단,2025 S/S 30수 라운드 T 반팔/긴팔 (성인/아동) 1P,의류 > 유아동의류 > 유아동의류,,,
5,0.495487,0.446340,1.0,0.0,2022/07/25,7.0,관공서/공공기관 > 경찰청/경찰서/소방서 > 경찰서 > 경찰서,-,[초특가 리큐엠] 넥밴드 선풍기 QNF100 화이트,디지털/가전 > 건강/냉난방가전 > 선풍기,,,
6,0.495399,0.446180,1.0,0.0,2023/07/13,7.0,자영업 > 소상공/자영업 > 놀이방/키즈카페 > 놀이방/키즈카페,LYCKA,옥스퍼드 보온가방 (23*14*20cm) 1P,일상용품 > 생활용품 > 생활잡화류,,,
7,0.494634,0.444788,1.0,0.0,2025/06/19,6.0,기업/회사 > 병원/의료기관 > 성형외과 > 성형외과,여수 조 성형외과,행잉세면 워시백 세면파우치 화장품파우치 여행 1P,일상용품 > 생활용품 > 생활잡화류,,,
8,0.492539,0.440980,1.0,0.0,2022/05/12,5.0,관공서/공공기관 > 보건소/정신건강/보건복지관련 > 금연지원센터/금연클리닉 > 금연지원센터/금연클리닉,충남금연지원센터,[스타벅스] 리유저블 텀블러 473ml 1P,일상용품 > 주방용품 > 저장용기,,,
9,0.490455,0.437190,1.0,0.0,2022/06/09,6.0,"관공서/공공기관 > 정부기관,관공서 > 각종 위원회 > 각종 위원회",국가인권위원회 광주인권사무소,목걸이형 에어플랫 휴대용선풍기 (허리버클기능),디지털/가전 > 건강/냉난방가전 > 선풍기,,,


## 10. 유사 거래 사례에서 추천 신호 추출

*RAG 단계: Retrieval 결과 재구성 — 검색된 거래 사례를 그대로 쓰지 않고,
"자주 등장한 상품분류/상품명/가격대/맥락"으로 요약해서 다음 단계(상품 랭킹)의 입력으로 만듭니다.
거래 상품명과 현재 상품명을 1:1로 매칭하지 않는다는 점이 핵심입니다.*

In [11]:
def weighted_value_counts(df, col, weight_col="trade_total_score", top_n=10):
    """단순 빈도가 아니라 거래 점수로 가중합을 낸 값 카운트입니다."""
    rows = [(clean_text(row.get(col, "")), float(row.get(weight_col, 0))) for _, row in df.iterrows() if clean_text(row.get(col, ""))]
    if not rows:
        return pd.DataFrame(columns=[col, "weighted_score", "count"])

    temp = pd.DataFrame(rows, columns=[col, "weight"])
    return (
        temp.groupby(col)
        .agg(weighted_score=("weight", "sum"), count=("weight", "count"))
        .reset_index()
        .sort_values("weighted_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


def extract_trade_signals(similar_trades: pd.DataFrame) -> dict:
    """유사 거래 사례들을 '신호 텍스트' 하나로 요약합니다. 이 텍스트를 다시 임베딩해서
    상품 후보를 찾는 데 씁니다 (거래 상품명 → 현재 상품명 직접 매칭이 아님)."""
    signal_cols = {
        "category_large": ("trade_category_large", "유사거래 상품분류 대"),
        "category_middle": ("trade_category_middle", "유사거래 상품분류 중"),
        "category_small": ("trade_category_small", "유사거래 상품분류 소"),
        "products": ("trade_product_name", "유사거래 상품명"),
        "buyers": ("buyer_type_middle", "유사구매처"),
        "events": ("event_filter", "행사"),
        "targets": ("target_filter", "대상"),
        "seasons": ("season_filter", "시즌"),
    }

    signals = {}
    signal_parts = []
    for key, (col, title) in signal_cols.items():
        counts = weighted_value_counts(similar_trades, col, top_n=10)
        signals[key] = counts
        top_values = counts[col].head(8).tolist() if len(counts) else []
        if top_values:
            signal_parts.append(f"[{title}] " + " ".join(top_values))

    signals["signal_text"] = clean_text("\n".join(signal_parts))
    return signals


signals = extract_trade_signals(similar_trades)

print("[거래 신호 텍스트]")
print(signals["signal_text"])

print("\n[상위 상품분류 중]")
display(signals["category_middle"])

[거래 신호 텍스트]
[유사거래 상품분류 대] 일상용품 디지털/가전 교육/문화용품 의류 [유사거래 상품분류 중] 생활용품 주방용품 건강/냉난방가전 문구/사무용품 노트북액세서리 유아동의류 [유사거래 상품분류 소] 저장용기 생활잡화류 선풍기 필기용품 이미용소품 노트북가방/케이스 유아동의류 [유사거래 상품명] [캐나믹스] CANAMIX M6 텀블러 600ml 1P [파카] 조터 오리지널 볼펜 (스페셜) 미니손톱깎이 (부엉이) 노트북 파우치 가방 14.1(39*28*4.0cm) 2025 S/S 30수 라운드 T 반팔/긴팔 (성인/아동) 1P [초특가 리큐엠] 넥밴드 선풍기 QNF100 화이트 옥스퍼드 보온가방 (23*14*20cm) 1P 행잉세면 워시백 세면파우치 화장품파우치 여행 1P [유사구매처] 경찰청/경찰서/소방서 청년관련센터/기관 제조/생산기업 부동산/분양/모델하우스 중고등학교 소상공/자영업 병원/의료기관 보건소/정신건강/보건복지관련

[상위 상품분류 중]


,trade_category_middle,weighted_score,count
0,생활용품,1.494098,3
1,주방용품,1.005574,2
2,건강/냉난방가전,0.985942,2
3,문구/사무용품,0.507466,1
4,노트북액세서리,0.497600,1
5,유아동의류,0.496401,1


## 11. 상품 후보 랭킹

*RAG 단계: Retrieval 결과 활용 — 거래 신호(문맥)와 질문 자체를 각각 현재 상품데이터와 임베딩 비교해서
최종 점수를 매깁니다. TF-IDF 키워드 매칭보다 "거래 맥락 + 실제 상품군 패턴"을 더 중요하게 봅니다.*

| 점수 | 비중 | 의미 |
|---|---:|---|
| query_product_score | 20% | 질문과 상품의 의미 유사도 |
| trade_signal_product_score | 30% | 유사 거래 신호와 상품의 의미 유사도 |
| trade_category_signal_score | 20% | 유사 거래 상품분류와 상품분류 일치 |
| budget_score | 15% | 예산 조건 충족 |
| quantity_score | 5% | 최소구매수량 조건 충족 |
| product_quality_score | 10% | 추천에 필요한 정보(가격/설명 등) 보유 여부 |

In [12]:
def calc_budget_score(price, budget):
    if budget in [None, ""] or pd.isna(budget):
        return 0.5
    if pd.isna(price):
        return 0.2
    price, budget = float(price), float(budget)
    if price <= budget:
        return 1.0
    if price <= budget * 1.2:
        return 0.65
    if price <= budget * 1.5:
        return 0.35
    return 0.0


def calc_quantity_score(moq, quantity):
    if quantity in [None, ""] or pd.isna(quantity):
        return 0.5
    if pd.isna(moq):
        return 0.7
    moq, quantity = float(moq), float(quantity)
    if moq <= quantity:
        return 1.0
    if moq <= quantity * 1.5:
        return 0.5
    return 0.0


def calc_product_quality_score(row):
    score = 0.0
    if clean_text(row.get("product_name", "")):
        score += 0.25
    if clean_text(row.get("category_path", "")):
        score += 0.25
    if not pd.isna(row.get("price", np.nan)):
        score += 0.25
    if clean_text(row.get("keywords", "")) or clean_text(row.get("summary", "")):
        score += 0.25
    return score


def calc_trade_category_signal_score(row, signals: dict) -> float:
    """거래데이터에서 자동 추출된 상품분류 신호가 현재 상품 텍스트에 얼마나 겹치는지 봅니다.
    수동 상품 사전이 아니라, 이번에 검색된 거래 사례에서 나온 분류를 그대로 씁니다."""
    product_text = combine_text(row.get("category_path", ""), row.get("product_name", ""), row.get("keywords", ""))

    score = 0.0
    for signal_key, col_name, weight in [
        ("category_middle", "trade_category_middle", 0.55),
        ("category_small", "trade_category_small", 0.35),
        ("category_large", "trade_category_large", 0.10),
    ]:
        signal_df = signals.get(signal_key)
        if signal_df is None or len(signal_df) == 0:
            continue
        max_weighted = signal_df["weighted_score"].max()
        for _, srow in signal_df.iterrows():
            value = clean_text(srow.get(col_name, ""))
            if value and value in product_text:
                score += (float(srow["weighted_score"]) / max_weighted if max_weighted else 0) * weight

    return min(score, 1.0)


def make_recommend_reason(row, budget, quantity) -> str:
    reasons = []
    if row["trade_signal_product_score"] >= 0.55:
        reasons.append("유사 거래 사례에서 나타난 상품군과 의미적으로 가까움")
    elif row["trade_signal_product_score"] >= 0.35:
        reasons.append("유사 거래 신호와 일부 관련 있음")

    if row["trade_category_signal_score"] >= 0.5:
        reasons.append("과거 유사 거래 상품분류와 현재 상품분류가 일치")
    elif row["trade_category_signal_score"] > 0:
        reasons.append("과거 유사 거래 상품분류와 일부 겹침")

    if row["query_product_score"] >= 0.55:
        reasons.append("질문과 상품 설명의 의미 유사도 높음")

    if budget not in [None, ""] and not pd.isna(budget) and pd.notna(row["price"]):
        reasons.append(f"예산 {int(float(budget)):,}원 이하 조건 충족" if float(row["price"]) <= float(budget) else "예산 초과 여부 확인 필요")

    if quantity not in [None, ""] and not pd.isna(quantity):
        moq_ok = pd.isna(row["moq"]) or float(row["moq"]) <= float(quantity)
        reasons.append("요청 수량 기준 최소구매수량 충족 가능" if moq_ok else "최소구매수량 확인 필요")

    return " / ".join(reasons) if reasons else "현재 상품데이터 기준 추천 후보로 검토 가능"


def rank_products(query: str, query_condition: dict, signals: dict, top_k: int = 5) -> pd.DataFrame:
    query_text = build_query_text(query_condition)

    query_product_scores = search_index(query_text, product_index)
    signal_text = signals.get("signal_text", "")
    trade_signal_product_scores = search_index(signal_text, product_index) if signal_text else np.zeros(len(product_df))

    result = product_df.copy()
    result["query_product_score"] = query_product_scores
    result["trade_signal_product_score"] = trade_signal_product_scores
    result["trade_category_signal_score"] = result.apply(lambda row: calc_trade_category_signal_score(row, signals), axis=1)

    budget = query_condition.get("budget_max")
    quantity = query_condition.get("quantity")
    result["budget_score"] = result["price"].apply(lambda p: calc_budget_score(p, budget))
    result["quantity_score"] = result["moq"].apply(lambda q: calc_quantity_score(q, quantity))
    result["product_quality_score"] = result.apply(calc_product_quality_score, axis=1)

    result["final_score"] = (
        result["query_product_score"] * 0.20 +
        result["trade_signal_product_score"] * 0.30 +
        result["trade_category_signal_score"] * 0.20 +
        result["budget_score"] * 0.15 +
        result["quantity_score"] * 0.05 +
        result["product_quality_score"] * 0.10
    )

    result["recommend_reason"] = result.apply(lambda row: make_recommend_reason(row, budget, quantity), axis=1)

    output_cols = [
        "product_id", "product_name", "price", "moq", "category_path", "keywords",
        "query_product_score", "trade_signal_product_score", "trade_category_signal_score",
        "budget_score", "quantity_score", "product_quality_score",
        "final_score", "recommend_reason", "image",
    ]
    return result.sort_values("final_score", ascending=False).head(top_k)[output_cols].reset_index(drop=True)

## 12. 최종 추천 함수

*RAG 단계 전체 통합 — 지금까지의 4단계(Query 이해 → Retrieval → 신호추출 → 랭킹)를
하나의 함수로 묶어서, 질문 하나로 전체 파이프라인을 실행할 수 있게 합니다.*

```text
질문 → 조건 추출(Query 이해) → 참고월 계산 → 유사 거래 검색(Retrieval) → 거래 신호 추출 → 상품 랭킹
```

In [13]:
def recommend_products_v1(query: str, top_k: int = DEFAULT_TOP_K, trade_top_k: int = 30) -> dict:
    query_condition = extract_query_condition(query)
    similar_trades = retrieve_similar_trades(query, query_condition, top_k=trade_top_k)
    signals = extract_trade_signals(similar_trades)
    recommendations = rank_products(query, query_condition, signals, top_k=top_k)

    return {
        "query": query,
        "query_condition": query_condition,
        "reference_month_weights": get_reference_month_weights(query_condition),
        "similar_trades": similar_trades,
        "signals": signals,
        "recommendations": recommendations,
        "backend": {
            "product": product_index["backend"],
            "trade": trade_index["backend"],
            "embedding_model": product_index.get("model"),
        },
    }


# 테스트
test_query = "8월 행사에서 나눠줄 여름 판촉물 추천해줘"
result = recommend_products_v1(test_query, top_k=5, trade_top_k=20)

print("[질문]", result["query"])
print("\n[추출된 조건]")
print(json.dumps(result["query_condition"], ensure_ascii=False, indent=2))
print("\n[사용 backend]", result["backend"])

print("\n[추천 상품]")
display(result["recommendations"])

[질문] 8월 행사에서 나눠줄 여름 판촉물 추천해줘

[추출된 조건]
{
  "raw_query": "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
  "buyer_context": null,
  "event_context": "여름 행사",
  "purpose": "판촉물",
  "season": "여름",
  "event_month": 8,
  "budget_max": null,
  "quantity": null,
  "style_keywords": [],
  "must_have": [],
  "extraction_method": "ollama_llm"
}

[사용 backend] {'product': 'ollama', 'trade': 'ollama', 'embedding_model': 'bge-m3'}

[추천 상품]


,product_id,product_name,price,moq,category_path,keywords,query_product_score,trade_signal_product_score,trade_category_signal_score,budget_score,quantity_score,product_quality_score,final_score,recommend_reason,image
0,81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품 >,"판촉물,사은품,기념품,답례품,선물용품,선물,홍보물,홍보용품,장갑,고무장갑,주방장갑,다용도장갑,위생장갑,멀티장갑,기모장갑,기모고무장갑,라텍스장갑",0.428466,0.579656,0.55,0.5,0.5,1.0,0.569590,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,231103G.jpg
1,84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월 >,#타올#수건#미용타올#미용실타올#극세사수건#극세사타올,0.412581,0.587158,0.55,0.5,0.5,1.0,0.568664,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,뷰티미용타올(메인).jpg
2,101669,[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈),4163.0,5000.0,가정/생활용품 > 치약칫솔세트외 > 치약/칫솔세트 >,#선물세트 #유시몰 #크리넥스 #칫솔치약세트 #물티슈,0.383974,0.593516,0.55,0.5,0.5,1.0,0.564850,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,thumb-7ze3dp7U2E81dF4EH2BYZrXkA7z72l_450x450.jpg
3,56194,민무늬(무지) 스카프 손수건 두건,1017.0,5000.0,가정/생활용품 > 수건(타월) > 손수건/스카프 >,"스카프,손수건,두건,등산용품,등산스카프,레져용품,패션두건,예쁜손수건,거즈,쁘띠손수건,등산용스카프,자전거스카프,단체스카프",0.409975,0.568419,0.55,0.5,0.5,1.0,0.562521,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,26/98900026.jpg
4,88025,직사각형8호 1단 송학보석함,73450.0,500.0,가정/생활용품 > 보석함/액세서리보관함 > 다용도보관함/트레이 >,"#나전철기 #자개보석함 #악세서리보관함,#다용도함,,#자개공예,#전통공예,#외국인선물,#답례품",0.442925,0.538830,0.55,0.5,0.5,1.0,0.560234,유사 거래 신호와 일부 관련 있음 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,썸네일(500)_121.jpg


## 13. 여러 질문 일괄 테스트 및 결과 저장

*RAG 단계: 평가 — 질문 여러 개로 파이프라인을 돌려 결과를 엑셀로 저장합니다 (품질 확인용).*

In [14]:
test_queries = [
    "8월 행사에서 나눠줄 여름 판촉물 추천해줘",
    "병원 개원 답례품으로 3천원 이하 500개 추천해줘",
    "대학교 OT에서 나눠줄 저렴한 사은품 추천해줘",
    "회사 창립기념품으로 실용적인 상품 추천해줘",
    "박람회 부스에서 나눠줄 홍보물 추천해줘",
    "12월 연말 고객 선물로 5만원 이하 상품 추천해줘",
    "어린이집 행사 답례품 추천해줘",
    "로고 인쇄 가능한 텀블러 추천해줘",
]

all_rows = []
for query in test_queries:
    r = recommend_products_v1(query, top_k=5, trade_top_k=20)
    query_condition = r["query_condition"]

    for rank, (_, row) in enumerate(r["recommendations"].iterrows(), start=1):
        all_rows.append({
            "query": query,
            "rank": rank,
            "buyer_context": query_condition.get("buyer_context", ""),
            "event_context": query_condition.get("event_context", ""),
            "purpose": query_condition.get("purpose", ""),
            "season": query_condition.get("season", ""),
            "event_month": query_condition.get("event_month", ""),
            "reference_month_weights": json.dumps(r["reference_month_weights"], ensure_ascii=False),
            "product_id": row["product_id"],
            "product_name": row["product_name"],
            "price": row["price"],
            "moq": row["moq"],
            "category_path": row["category_path"],
            "final_score": row["final_score"],
            "recommend_reason": row["recommend_reason"],
        })

batch_result_df = pd.DataFrame(all_rows)
batch_result_path = OUTPUT_DIR / "giftco_product_recommendation_v1_rebuild_260723_result.xlsx"
batch_result_df.to_excel(batch_result_path, index=False)

print("결과 저장:", batch_result_path)
display(batch_result_df.head(20))

결과 저장: output\giftco_product_recommendation_v1_rebuild_260723_result.xlsx


,query,rank,buyer_context,event_context,purpose,season,event_month,reference_month_weights,product_id,product_name,price,moq,category_path,final_score,recommend_reason
0,8월 행사에서 나눠줄 여름 판촉물 추천해줘,1,NaN,여름 행사,판촉물,여름,8.0,"{""5"": 1.0, ""4"": 0.85, ""6"": 1.0, ""3"": 0.25, ""7"": 1.0, ""8"": 0.45}",81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품 >,0.569590,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치
1,8월 행사에서 나눠줄 여름 판촉물 추천해줘,2,NaN,여름 행사,판촉물,여름,8.0,"{""5"": 1.0, ""4"": 0.85, ""6"": 1.0, ""3"": 0.25, ""7"": 1.0, ""8"": 0.45}",84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월 >,0.568664,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치
2,8월 행사에서 나눠줄 여름 판촉물 추천해줘,3,NaN,여름 행사,판촉물,여름,8.0,"{""5"": 1.0, ""4"": 0.85, ""6"": 1.0, ""3"": 0.25, ""7"": 1.0, ""8"": 0.45}",101669,[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈),4163.0,5000.0,가정/생활용품 > 치약칫솔세트외 > 치약/칫솔세트 >,0.564850,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치
3,8월 행사에서 나눠줄 여름 판촉물 추천해줘,4,NaN,여름 행사,판촉물,여름,8.0,"{""5"": 1.0, ""4"": 0.85, ""6"": 1.0, ""3"": 0.25, ""7"": 1.0, ""8"": 0.45}",56194,민무늬(무지) 스카프 손수건 두건,1017.0,5000.0,가정/생활용품 > 수건(타월) > 손수건/스카프 >,0.562521,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치
4,8월 행사에서 나눠줄 여름 판촉물 추천해줘,5,NaN,여름 행사,판촉물,여름,8.0,"{""5"": 1.0, ""4"": 0.85, ""6"": 1.0, ""3"": 0.25, ""7"": 1.0, ""8"": 0.45}",88025,직사각형8호 1단 송학보석함,73450.0,500.0,가정/생활용품 > 보석함/액세서리보관함 > 다용도보관함/트레이 >,0.560234,유사 거래 신호와 일부 관련 있음 / 과거 유사 거래 상품분류와 현재 상품분류가 일치
5,병원 개원 답례품으로 3천원 이하 500개 추천해줘,1,병원,개원,답례품,NaN,NaN,{},81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품 >,0.594448,"유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 일부 겹침 / 예산 3,000원 이하 조건 충족 / 최소구매수량 확인 필요"
6,병원 개원 답례품으로 3천원 이하 500개 추천해줘,2,병원,개원,답례품,NaN,NaN,{},56036,에코밴드세트 소D (95*66*16mm),1187.0,30000.0,가정/생활용품 > 응급 / 안전용품 > 밴드세트 >,0.586503,"유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 일부 겹침 / 예산 3,000원 이하 조건 충족 / 최소구매수량 확인 필요"
7,병원 개원 답례품으로 3천원 이하 500개 추천해줘,3,병원,개원,답례품,NaN,NaN,{},84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월 >,0.585677,"유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 일부 겹침 / 예산 3,000원 이하 조건 충족 / 최소구매수량 확인 필요"
8,병원 개원 답례품으로 3천원 이하 500개 추천해줘,4,병원,개원,답례품,NaN,NaN,{},56194,민무늬(무지) 스카프 손수건 두건,1017.0,5000.0,가정/생활용품 > 수건(타월) > 손수건/스카프 >,0.584575,"유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 일부 겹침 / 예산 3,000원 이하 조건 충족 / 최소구매수량 확인 필요"
9,병원 개원 답례품으로 3천원 이하 500개 추천해줘,5,병원,개원,답례품,NaN,NaN,{},3814,메탈레드 30매 물티슈,263.0,100000.0,가정/생활용품 > 물티슈/휴지 > 30매 이상 물티슈 >,0.583196,"유사 거래 신호와 일부 관련 있음 / 과거 유사 거래 상품분류와 일부 겹침 / 예산 3,000원 이하 조건 충족 / 최소구매수량 확인 필요"


## 14. Ollama로 고객용 추천 답변 생성

*RAG 단계: Generation — 랭킹/점수 계산은 코드가 이미 끝냈고, LLM은 그 결과를 자연스러운 문장으로만 바꿉니다.*

```text
- 추천 후보와 유사 거래 신호 안에 있는 내용만 사용
- 가격/수량/납기 등 데이터에 없는 내용은 단정하지 않기
- 추천 이유는 점수 항목을 기반으로 작성
```

In [15]:
def build_answer_context(result: dict) -> str:
    """LLM에게 넘길 컨텍스트(=Augmentation 단계 결과물)를 하나의 텍스트로 구성합니다."""
    lines = ["[질문]", result["query"], ""]
    lines += ["[추출된 조건]", json.dumps(result["query_condition"], ensure_ascii=False), ""]
    lines += ["[참고 주문월 가중치]", json.dumps(result["reference_month_weights"], ensure_ascii=False), ""]
    lines += ["[거래데이터 신호]", result["signals"].get("signal_text", ""), ""]

    lines.append("[추천 상품 후보]")
    for i, (_, row) in enumerate(result["recommendations"].iterrows(), start=1):
        lines += [
            f"{i}. {row['product_name']}",
            f"   - 가격: {row['price']}",
            f"   - 최소구매수량: {row['moq']}",
            f"   - 카테고리: {row['category_path']}",
            f"   - 추천근거: {row['recommend_reason']}",
            "",
        ]

    return "\n".join(lines)


def generate_customer_answer(query: str, model: str = LLM_MODEL, top_k: int = 5) -> dict:
    import ollama  # 없으면 여기서 ModuleNotFoundError 발생 (fallback 없음 — Generation 단계는 LLM이 필수)

    result = recommend_products_v1(query, top_k=top_k, trade_top_k=30)
    context = build_answer_context(result)

    system_prompt = """
너는 기프트코 판촉물 상품 추천 어시스턴트다.

규칙:
- 제공된 추천 후보와 거래데이터 신호만 근거로 답변한다.
- 데이터에 없는 납기, 재고, 인쇄 가능 여부는 단정하지 않는다.
- 가격과 최소구매수량은 제공된 값만 말한다.
- 고객에게 보여줄 수 있게 자연스러운 한국어로 답변한다.
- 상위 3~5개 상품을 추천하고, 각 상품의 추천 이유를 짧게 적는다.
- 마지막에 확인이 필요한 조건을 적는다.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": f"아래 근거를 바탕으로 고객에게 상품 추천 답변을 작성해줘.\n\n{context}"},
        ],
        options={"temperature": 0.2, "num_ctx": 4096},
        stream=False,
    )

    return {"answer": response["message"]["content"], "result": result}


# 사용 예시
answer_result = generate_customer_answer("8월 행사에서 나눠줄 여름 판촉물 추천해줘", top_k=5)
print(answer_result["answer"])
display(answer_result["result"]["recommendations"])

8월 여름 행사 판촉물로 딱 맞는 상품들을 추천해 드릴게요! 

1.  **극세사 뷰티미용 타올** (가격: 1582.0, 최소 구매 수량: 5000.0)
    *   추천 이유: 여름 행사와 잘 어울리는 극세사 타올은 휴대용으로도 좋고, 행사 후에도 실용적인 선물이 될 수 있습니다. 유사 거래 상품분류에서 타월 관련 상품이 많이 판매되고 있어 적합합니다.

2.  **[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈)** (가격: 4163.0, 최소 구매 수량: 5000.0)
    *   추천 이유: 여름철 위생용품 세트는 행사 선물로 좋은 선택입니다. 치약, 칫솔, 가글 등 다양한 제품이 포함되어 있어 실용성을 높일 수 있습니다.

3.  **민무늬(무지) 스카프 손수건 두건** (가격: 1017.0, 최소 구매 수량: 5000.0)
    *   추천 이유: 시원한 여름에 햇빛을 가려주는 스카프나 손수건은 실용적인 선물입니다. 다양한 색상으로 제작하여 행사의 분위기를 더할 수 있습니다.

4.  **[캐나믹스] CANAMIX M6 텀블러 600ml 1P** (가격: 1458.0, 최소 구매 수량: 10000.0)
    *   추천 이유: 여름철 수분 섭취를 권장하는 행사에서 텀블러는 유용한 선물이 될 수 있습니다. 

5.  **직사각형8호 1단 송학보석함** (가격: 73450.0, 최소 구매 수량: 500.0)
    *   추천 이유: 여름 행사에서 고급스러운 느낌을 더하고 싶다면 보석함은 좋은 선택이 될 수 있습니다.

**확인해야 할 사항:**

*   각 상품의 재고 수량은 현재 확인되지 않았습니다.
*   인쇄 가능 여부 (로고, 문구 등)는 별도로 확인 필요합니다.


,product_id,product_name,price,moq,category_path,keywords,query_product_score,trade_signal_product_score,trade_category_signal_score,budget_score,quantity_score,product_quality_score,final_score,recommend_reason,image
0,81981,[고급형] 미끄럼방지형 천연라텍스 따스한기모 고무장갑,1458.0,10000.0,가정/생활용품 > 청소용품 > 청소용품 >,"판촉물,사은품,기념품,답례품,선물용품,선물,홍보물,홍보용품,장갑,고무장갑,주방장갑,다용도장갑,위생장갑,멀티장갑,기모장갑,기모고무장갑,라텍스장갑",0.428466,0.587602,0.55,0.5,0.5,1.0,0.571974,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,231103G.jpg
1,84580,극세사 뷰티미용 타올,1582.0,5000.0,가정/생활용품 > 수건(타월) > 스포츠/극세사타월 >,#타올#수건#미용타올#미용실타올#극세사수건#극세사타올,0.412581,0.590407,0.55,0.5,0.5,1.0,0.569638,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,뷰티미용타올(메인).jpg
2,101669,[유시몰] 유덴트세트 6호(치약(20g)+알칫솔+가글(15ml)+크리넥스쿨링물티슈),4163.0,5000.0,가정/생활용품 > 치약칫솔세트외 > 치약/칫솔세트 >,#선물세트 #유시몰 #크리넥스 #칫솔치약세트 #물티슈,0.383974,0.592882,0.55,0.5,0.5,1.0,0.564660,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,thumb-7ze3dp7U2E81dF4EH2BYZrXkA7z72l_450x450.jpg
3,88025,직사각형8호 1단 송학보석함,73450.0,500.0,가정/생활용품 > 보석함/액세서리보관함 > 다용도보관함/트레이 >,"#나전철기 #자개보석함 #악세서리보관함,#다용도함,,#자개공예,#전통공예,#외국인선물,#답례품",0.442925,0.552432,0.55,0.5,0.5,1.0,0.564315,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,썸네일(500)_121.jpg
4,56194,민무늬(무지) 스카프 손수건 두건,1017.0,5000.0,가정/생활용품 > 수건(타월) > 손수건/스카프 >,"스카프,손수건,두건,등산용품,등산스카프,레져용품,패션두건,예쁜손수건,거즈,쁘띠손수건,등산용스카프,자전거스카프,단체스카프",0.409975,0.574257,0.55,0.5,0.5,1.0,0.564272,유사 거래 사례에서 나타난 상품군과 의미적으로 가까움 / 과거 유사 거래 상품분류와 현재 상품분류가 일치,26/98900026.jpg


# 다음 단계

이 v1(row 단위)을 실행한 뒤 확인할 것:

1. Ollama 임베딩이 정상적으로 사용되고 있는가 (TF-IDF fallback으로 빠지지 않았는가)?
2. LLM이 질문을 구조화한 결과(`query_condition`)가 실제 질문 의도와 맞는가?
3. 리드타임 가중치(참고 주문월)가 실제 거래데이터 분포와 맞아떨어지는가?
4. 유사 거래 사례에서 뽑은 "신호"가 실제로 관련 있는 상품군을 가리키는가?
5. 최종 추천 상품이 예산/수량 조건을 잘 지키는가?

## 다음 버전

다음: v2에서 거래데이터를 구매처/고객 profile 단위로 집계해서 검색합니다. 전체 버전 로드맵과 배경은 `README.md`를 참고하세요.
